# מפגש 22 — פתרון מורה Audited
תואם באופן מלא לנתוני מעבדת התלמיד, לא להדגמה: bill_length_mm, bill_depth_mm, seed=17, K מועמדים [1,3,5,15,51], חמישה קיפולים, macro-F1. מוצג למורה בלבד.

## חישוב ידני בסיסי
(0.8+0.9+0.7)/3=0.8, לא תוצאת Penguins. Train פנימי לומד, Validation פנימי בוחר ו־Test חיצוני נשמר לסוף.

In [ ]:
import numpy as np
assert np.isclose(np.mean([.8,.9,.7]),.8)
assert np.isclose(sum([.8,.9,.7]),2.4)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df=pd.read_csv(url)
assert {"species","flipper_length_mm","body_mass_g","bill_length_mm","bill_depth_mm"}.issubset(df.columns)
assert df.shape[0]==344
features=["bill_length_mm","bill_depth_mm"]
data=df.dropna(subset=["species"]).copy()
X=data[features].copy()
y=data["species"].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=17,stratify=y)
assert len(X_train)==258
assert len(X_test)==86
assert X_train.index.is_unique
assert X_test.index.is_unique
assert set(X_train.index).isdisjoint(set(X_test.index))
assert len(X_train)+len(X_test)==len(y)
assert X_train.columns.tolist()==features
print("Outer split:",X_train.shape,X_test.shape)

## בדיקות קיפולים: כל רשומה מופיעה פעם אחת כאימות פנימי.

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
seen=np.zeros(len(X_train),dtype=int)
for fold_number,(inner_train,inner_validation) in enumerate(cv.split(X_train,y_train),start=1):
    assert set(inner_train).isdisjoint(inner_validation)
    assert len(inner_train)+len(inner_validation)==len(X_train)
    assert set(inner_train).union(inner_validation)==set(range(len(X_train)))
    assert set(y_train.iloc[inner_validation])==set(y_train.unique())
    seen[inner_validation]+=1
    print("Fold",fold_number,"train",len(inner_train),"validation",len(inner_validation))
assert np.all(seen==1)
print("All outer Train rows validated once inside CV")

## הדגמת מניעת זליגה (לא נתוני מקור Penguins)

In [ ]:
# ניסוי סינתטי להמחשת מניעת זליגת מידע; אינו חלק מציוני Penguins
tiny_train=np.array([[0.0],[2.0],[4.0]])
tiny_validation=np.array([[100.0]])
safe_scaler=StandardScaler().fit(tiny_train)
unsafe_scaler=StandardScaler().fit(np.vstack([tiny_train,tiny_validation]))
assert np.isclose(safe_scaler.mean_[0],2.0)
assert np.isclose(unsafe_scaler.mean_[0],26.5)
assert not np.isclose(safe_scaler.mean_[0],unsafe_scaler.mean_[0])
assert np.isclose(safe_scaler.transform(tiny_validation)[0,0],(100-2)/safe_scaler.scale_[0])
print("Training-only mean:",safe_scaler.mean_[0],"leaked mean:",unsafe_scaler.mean_[0])

## חמשת מועמדי K

In [ ]:
candidates=[1,3,5,15,51]
metric="f1_macro"
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
rows=[]
fold_detail={}
for k in candidates:
    pipeline=make_pipeline(SimpleImputer(strategy="mean"),StandardScaler(),
                           KNeighborsClassifier(n_neighbors=k,weights="uniform",metric="euclidean"))
    fold=cross_validate(pipeline,X_train,y_train,cv=cv,scoring=metric,
                        return_train_score=True,n_jobs=1)
    assert len(fold["test_score"])==5
    assert len(fold["train_score"])==5
    assert np.isfinite(fold["test_score"]).all()
    fold_detail[k]=fold["test_score"].copy()
    rows.append({"K":k,"train_mean":fold["train_score"].mean(),
                 "cv_mean":fold["test_score"].mean(),
                 "cv_std":fold["test_score"].std(ddof=0)})
results_df=pd.DataFrame(rows)
assert results_df["K"].tolist()==candidates
assert len(results_df)==5
assert results_df["cv_mean"].between(0,1).all()
assert results_df["cv_std"].ge(0).all()
assert all(len(scores)==5 for scores in fold_detail.values())
display(results_df)

## פירוש תוצאות, בלי לחפש ציון Test

In [ ]:
assert results_df["train_mean"].between(0,1).all()
assert results_df["cv_mean"].between(0,1).all()
assert results_df["cv_std"].ge(0).all()
assert all(results_df["K"].value_counts()==1)
assert all(len(fold_detail[k])==5 for k in candidates)
print(results_df)

## התאמה מחדש והערכה יחידה

In [ ]:
ranked=results_df.sort_values(["cv_mean","K"],ascending=[False,True])
best_k=int(ranked.iloc[0]["K"])
assert best_k in candidates
assert np.isclose(ranked.iloc[0]["cv_mean"],results_df["cv_mean"].max())
assert results_df.loc[results_df["K"]==best_k,"cv_mean"].size==1
final_model=make_pipeline(SimpleImputer(strategy="mean"),StandardScaler(),
                          KNeighborsClassifier(n_neighbors=best_k,weights="uniform",metric="euclidean"))
final_model.fit(X_train,y_train)
assert final_model.named_steps["kneighborsclassifier"].n_neighbors==best_k
assert final_model.named_steps["standardscaler"].n_features_in_==2
predictions=final_model.predict(X_test)
assert len(predictions)==len(y_test)==86
test_f1=f1_score(y_test,predictions,average="macro")
test_acc=accuracy_score(y_test,predictions)
assert 0<=test_f1<=1
assert 0<=test_acc<=1
report=classification_report(y_test,predictions,labels=["Adelie","Chinstrap","Gentoo"],output_dict=True,zero_division=0)
assert sum(report[x]["support"] for x in ["Adelie","Chinstrap","Gentoo"])==86
print("Chosen by inner CV:",best_k,"Outer Test macro-F1:",test_f1,"Accuracy:",test_acc)

## בדיקות תוצאות הדיווח

In [ ]:
assert all(0<=report[k]["precision"]<=1 for k in ["Adelie","Chinstrap","Gentoo"])
assert all(0<=report[k]["recall"]<=1 for k in ["Adelie","Chinstrap","Gentoo"])
assert all(0<=report[k]["f1-score"]<=1 for k in ["Adelie","Chinstrap","Gentoo"])
assert report["macro avg"]["support"]==86
assert abs(report["macro avg"]["f1-score"]-test_f1)<1e-12
assert abs(accuracy_score(y_test,predictions)-test_acc)<1e-12
print("Teacher audit assertions passed")

## פתרון מילולי למורה
בתוך cross_validate ה־test_score הוא ממוצע האימות הפנימי, **לא** Outer Test. הערכת Test אינה משמשת להחלפת K. פערים בין תוצאות קיפולים אינם רווח סמך; פיצול קבוצות מחייב GroupKFold/חלוקה לפי קבוצה בתכנון מתאים. השוואת דוגמת מורה למעבדה על seeds ומאפיינים שונים אינה ניסוי מבוקר.